# Contextual Compression + Ensemble Retriever — 최고 품질 검색 조합

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

- **Ensemble**과 **Contextual Compression**을 결합하는 이유와 효과 이해하기
- 두 기법을 순서대로 연결하는 파이프라인 구현하기
- 실제 프로덕션 환경에서 권장하는 검색 파이프라인 설계하기

## 사전 지식

- 03-EnsembleRetriever: BM25 + Vector 결합
- 02-ContextualCompressionRetriever: 문서 압축 및 필터링

---

```mermaid
flowchart LR
    Q[사용자 쿼리]:::input --> B[BM25<br/>키워드 검색]:::process
    Q --> V[FAISS<br/>의미 검색]:::process
    B --> E[Ensemble<br/>다양한 문서 수집]:::process
    V --> E
    E --> F[EmbeddingsFilter<br/>관련성 필터링]:::process
    F --> R[고품질<br/>압축 결과]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

## 결합 전략

### 1단계: Ensemble Retriever
- BM25(키워드) + FAISS(의미) 검색
- 하이브리드 검색으로 다양한 후보 문서 수집

### 2단계: Contextual Compression
- 수집된 문서에서 관련 내용만 필터링
- 노이즈 제거, 품질 향상

> **실무 팁**: `EmbeddingsFilter`는 LLM 호출 없이 빠르게 작동해서 프로덕션 환경에 적합해요. 최고 품질이 필요하다면 `LLMChainExtractor`를 추가하세요.

> **주의**: 이 노트북을 실행하기 전에 03-EnsembleRetriever와 02-ContextualCompressionRetriever를 먼저 학습하세요.

> **실무 팁**: Ensemble의 k를 Compression 이후에 원하는 최종 문서 수보다 넉넉히 설정하세요. 예를 들어 최종 3개 문서가 필요하다면 k=8~10으로 설정하여 Compression 단계에서 선택할 여지를 충분히 줍니다.

In [ ]:
from dotenv import load_dotenv
load_dotenv()


In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

# ---------------------------------------------------
# Ensemble + Compression 파이프라인 생성
# ---------------------------------------------------

# ============================================================
# TODO: BM25와 FAISS를 EnsembleRetriever로 결합하고 EmbeddingsFilter 압축을 추가하세요
# 힌트: EnsembleRetriever → base_retriever로, EmbeddingsFilter → base_compressor로 전달
# 예상 결과: Ensemble + Compression 검색기 생성 완료 메시지 출력
# ============================================================

from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever, ContextualCompressionRetriever
from langchain.retrievers.document_compressors import EmbeddingsFilter
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 준비

# 1. BM25 Retriever (키워드 기반)
# k=5: 압축 후 줄어들 것을 감안해 넉넉히 검색

# 2. FAISS Retriever (의미 기반)

# 3. Ensemble Retriever — RRF로 BM25와 FAISS 결과를 통합

# 4. EmbeddingsFilter — LLM 호출 없이 임베딩 유사도로 관련 없는 문서 제거

# 5. 최종 결합 — Ensemble로 다양하게 수집, Filter로 품질 확보


In [ ]:
# 검색 및 비교

# Ensemble만

# Ensemble + Compression

# 아래에 코드를 작성하세요


---

## 마무리 정리

### 핵심 요약

| 항목 | 내용 |
|------|------|
| 핵심 아이디어 | BM25(Sparse) + FAISS(Dense) → Ensemble → EmbeddingsFilter(압축) |
| 단계별 역할 | Ensemble: 다양한 관점 수집 / Compression: 노이즈 제거 |
| 압축기 선택 | 비용 민감 → EmbeddingsFilter / 정확도 중시 → LLMChainExtractor |
| 성능 vs 비용 | 단계가 많을수록 품질 향상, LLM 호출 비용 증가 |
| 한국어 최적화 | BM25 부분에 KiwiBM25Retriever 교체 시 한국어 성능 추가 향상 |

### 6-5 Retriever 시리즈 완성 — 전체 비교

| Retriever | 핵심 강점 | 주요 비용 |
|-----------|-----------|-----------|
| VectorStore | 의미 기반, 빠름 | 임베딩 |
| ContextualCompression | 노이즈 제거 | LLM 호출 |
| Ensemble | Sparse+Dense 결합 | 없음 (결합만) |
| ParentDocument | 정밀도+컨텍스트 | 메모리 |
| MultiQuery | 쿼리 다각화 | LLM 호출 |
| MultiVector | 요약 기반 검색 | 인덱싱 LLM |
| SelfQuery | 메타데이터 필터 | LLM 파싱 |
| TimeWeighted | 최신성 반영 | 없음 |
| KiwiBM25 | 한국어 키워드 | 형태소 분석 |
| **CC+Ensemble** | **모든 강점 결합** | **중간 수준** |

### 다음 섹션 예고

**6-6 Reranker** — 검색 후 순위 재조정(리랭킹) 기법으로 최종 품질을 한 단계 더 끌어올리는 방법을 배워요. Cross-Encoder, Cohere, Jina 세 가지 리랭커를 다뤄요.